# Lesson 12 — Learn which token arrangements belong together

**Learning objective:** separate learning a visual vocabulary from learning a sampling distribution.

A vocabulary tells us what symbols mean. A prior learns which symbol should
come next, given the symbols already present. Scanning the token grid row by
row lets a small GRU learn these dependencies while the VQ encoder and decoder
stay frozen. Training supplies the true previous tokens; sampling supplies the
prior's own previous choices.

## Setup

Run this cell first. In Colab it fetches the course code automatically;
locally it uses your checkout. The lessons use short CPU experiments.

In [ ]:
import os
from pathlib import Path
import subprocess
import sys

# Find an existing checkout before fetching one in a fresh Colab session.
_start = Path.cwd().resolve()
_course_root = next((
    candidate for candidate in (_start, *_start.parents)
    if (candidate / "course/curriculum.yaml").is_file()
    and (candidate / "src/course_aiml_autoencoders").is_dir()
), None)
if _course_root is None:
    try:
        import google.colab
    except ImportError as error:
        raise RuntimeError("Open this notebook from the course checkout or in Google Colab.") from error
    _course_root = _start / "course-aiml-autoencoders"
    if not _course_root.exists():
        subprocess.run([
            "git", "clone", "--depth", "1", "--branch", "main",
            "https://github.com/tsilva/course-aiml-autoencoders.git",
            str(_course_root),
        ], check=True)
    if not (_course_root / "course/curriculum.yaml").is_file():
        raise RuntimeError(f"Incomplete course checkout at {_course_root}; rename it and rerun setup.")
os.chdir(_course_root)
_course_src = str(_course_root / "src")
if _course_src not in sys.path:
    sys.path.insert(0, _course_src)
print("Course ready:", _course_root)

In [ ]:
import matplotlib.pyplot as plt
import torch
from IPython.display import Image, display
from course_aiml_autoencoders.config import load_yaml, with_overrides
from course_aiml_autoencoders.course import (
    balanced_class_batch, learn, learn_prior, lesson_config,
    load_metrics, load_run_summary, load_trained_model,
    plot_metric_history, repository_root,
)
from course_aiml_autoencoders.data import build_dataloaders, class_names
from course_aiml_autoencoders.diagnostics import (
    per_example_mse, plot_image_grid, plot_reconstruction_grid,
)

ROOT = repository_root()
torch.manual_seed(0)
PROFILE = "quick"

The default uses eight CPU epochs on 2,048 training images and 512 held-out
images. A matching completed run is reused automatically; its exact path and
measured duration are printed. These short runs expose mechanisms, not settled
rankings. The first Fashion-MNIST lesson downloads the dataset once.

Set `PROFILE = "full"` for the original training budget; `learn(..., rerun=True)`
creates fresh evidence. You can return to the prediction while training runs.

## Predict before running

Will beating a uniform token predictor prove that the prior learned spatial dependencies?

<details>
<summary>Reveal the expected reasoning</summary>

No. Frequent symbols alone can beat uniform prediction. Compare with a training-frequency unigram baseline as well as inspecting generated arrangements.
</details>

In [ ]:
vq_run = learn("recipes/vqvae/vqvae-001-basic.yaml", profile=PROFILE)
prior_run = learn_prior("recipes/prior/prior-001-gru.yaml", vq_run, profile=PROFILE)
summary = load_run_summary(prior_run)
print("Uniform CE:", summary["uniform_cross_entropy"],
      "training-frequency unigram CE:", summary["unigram_cross_entropy"],
      "learned-prior validation CE:", summary["best_validation_cross_entropy"])
_ = plot_metric_history(load_metrics(prior_run), ["train/cross_entropy", "validation/cross_entropy"])
display(Image(filename=str(vq_run / "figures/uniform-random-token-samples.png")))
display(Image(filename=str(prior_run / "figures/learned-prior-samples.png")))

Better validation likelihood than the unigram baseline supports learning
structure beyond overall symbol frequency, but is not proof of good generated
images. Inspect arrangements and diversity: sampling consumes its own history
and can accumulate errors. Short training may reveal only frequency learning.

Prior perplexity measures next-token uncertainty given history; codebook
perplexity measures assignment diversity. They share a name but answer
different questions. The prior cannot restore information discarded by the
frozen VQ representation.

## Advancement gate — transfer check

The prior improves dramatically, but generated images still lack fine detail. Which stage might be the bottleneck?

<details>
<summary>Reveal the expected reasoning</summary>

The VQ encoder/codebook/decoder may have discarded that detail. A better token distribution cannot recover information absent from the representation.
</details>

<details>
<summary>Optional: go deeper</summary>

Raster order gives the factorization
$p(k_1,\ldots,k_N)=\prod_i p(k_i\mid k_{<i})$.
Teacher forcing uses true history during training; sampling uses sampled
history. The small GRU is chosen for clarity, not top-tier image quality.
Temperature and larger priors are optional investigations; preserve the exact
paired VQ checkpoint when comparing them.
</details>

**Next:** [Lesson 13](13-synthesis.ipynb) · [Open in Colab](https://colab.research.google.com/github/tsilva/course-aiml-autoencoders/blob/main/course/notebooks/13-synthesis.ipynb). No worksheet is required.